# ECON2041 Week 9 tutorial: threats to causality

### What you'll be able to do by the end

- Upload a CSV file to Colab, as you'll do in the problem set
- Read a p-value from a regression table
- Name the threat to a causal reading of a correlation: selection bias or reverse causality
- Predict the sign of the bias from a factor that a regression leaves out

### Different types of cells

- 🟢 Read and run: these cells have pre-written code, you can simply run them, read and interpret the output
- ✏️ You write: your turn to try to write code or sometimes add a short written answer
- 🤔 You think: stop and think on your own before running the next cell or reading on
- 💬 Discuss: talk with your neighbors before typing or running

## Setup

Our usual setup block, plus two extra lines, `glob` and `os`, which we'll use in the upload cell below to find your CSV file.

In [ ]:
# 🟢 Read and run: our standard ECON2041 setup block
import numpy as np               # numerical tools (nicknamed np)
import pandas as pd              # data tools (nicknamed pd)
import matplotlib.pyplot as plt  # plotting tools (nicknamed plt)
import seaborn as sns            # statistical charts (nicknamed sns)
from statsmodels.formula.api import ols  # ordinary least squares regression
import glob                      # finds files by their name
import os                        # file tools

# Keep scalar output plain under NumPy 2 (0.5, not np.float64(0.5)).
if np.lib.NumpyVersion(np.__version__) >= "2.0.0":
    np.set_printoptions(legacy="1.25")

DATA = "https://emiliatjernstrom.com/econ2041/data"   # Unit datasets live at this web address

print("Setup done!")

## Upload a CSV file to Colab: practice for the problem set

1. Download `econ2041-recordings-quiz.csv` from the week 9 tutorial page on iLearn
2. Click the folder icon in Colab's left sidebar to open the Files panel
3. Drag the CSV file into that panel, or click the upload icon at the top of the panel and find the file wherever your browser saved it
4. Run the cell below

Colab forgets uploaded files when the runtime restarts, so if you come back to your notebook after leaving it for a while, you may need to re-upload the CSV file and run the setup cell again.

In [ ]:
# 🟢 Read and run: load econ2041-recordings-quiz.csv into a dataframe called recordings_quiz
# look for the CSV file in Colab's Files panel, newest first
found = set(
    glob.glob("econ2041-recordings-quiz*.csv")
    + glob.glob("/content/econ2041-recordings-quiz*.csv")
    + glob.glob("/econ2041-recordings-quiz*.csv")
)

if found:
    csv = max(found, key=os.path.getmtime)
    recordings_quiz = pd.read_csv(csv)
    print("Loaded", csv)
    display(recordings_quiz.head())
else:
    print("No econ2041-recordings-quiz CSV file yet. Please follow the instructions above to upload the dataset.")

### What's in this dataset

The data cover ECON2041 students this semester: one row for each of the 209 students who sat the week 7 quiz and appear in the Echo360 viewing records.

- `recordings`: how many of the six live lecture recordings from weeks 1 to 6 the student watched at least part of on Echo360
- `hours`: the total time that the student spent watching those six recordings on Echo360, in hours
- `quiz`: the student's week 7 quiz mark out of 20

The data are anonymous. To make it extra impossible for anyone to be identified in the data, we...

- removed all names
- removed all student IDs, 
- put the rows in random order, and 
- added a small random number to every quiz mark (most marks in the file are within 1 of the real mark, but they are not identical)
  
None of these steps changes the conclusions in today's questions.

*Source:* Echo360 viewing records for the ECON2041 live lectures, from iLearn Insights, and the ECON2041 week 7 quiz marks.

## Live lecture recordings and the quiz

### Question 1: do the live lecture recordings raise quiz marks?

The dataset that you just uploaded is stored in `recordings_quiz`.
The causal question: does watching the live lecture recordings raise quiz marks?
The dataset measures watching in two ways, `recordings` and `hours`, so we'll start with how the two measures are related, and then use one of them in a regression.

In [ ]:
# ✏️ You write: a scatterplot of hours (vertical) against recordings (horizontal), one dot per student
# Hint: sns.scatterplot with data=recordings_quiz, x and y, plus s=20 and alpha=0.5

In [ ]:
# 🟢 Read and run: the correlation between recordings and hours
recordings_quiz[["recordings", "hours"]].corr().round(2)

🤔 Did students who watched more recordings also spend more hours watching?
How much do hours vary among students who watched the same number of recordings?

In [ ]:
# ✏️ You write: run a regression of quiz on hours, store the result as hours_fit, and show the coefficients rounded to 2 decimals
# Hint: ols() takes a formula with the outcome to the left of ~ and the explanatory variable to the right,
#       plus data=recordings_quiz; .fit() runs the regression, and .params holds the coefficients

✏️ Complete the sentence:

- Each additional hour of live lecture recordings watched is associated with a quiz mark that is ________ marks higher, on average

In [ ]:
# 🟢 Read and run: the regression table for hours_fit, rounded to 3 decimals
hours_fit.summary2().tables[1].round(3)

The table has one row per coefficient, `Intercept` and `hours`, and one column per statistic:

- `Coef.`: the same coefficients as `.params` above
- `P>|t|`: the p-value for each coefficient, for the null hypothesis that the coefficient is 0
- `Std.Err.`, `t`, `[0.025`, and `0.975]`: we'll practice these in the week 10 tutorial

We saw p-values in the week 5 Essential Concepts recordings: a p-value is the probability of an estimate at least this far from the null value if the null hypothesis were true.
In the week 9 Essential Concepts recordings, we applied them to regression.
Remember that in economics, we typically reject the null hypothesis when the p-value is below 0.1.

In [ ]:
# ✏️ You write: show the p-values of hours_fit, rounded to 3 decimals
# Hint: the p-values are stored in .pvalues, as the coefficients are stored in .params

🤔 Is the p-value on `hours` below 0.1?
What does that tell us about the slope of 0.92?

The slope of 0.92 is an association between hours watched and quiz marks.
Is it the causal effect of watching?
In the week 8 Essential Concepts recordings, we named three threats to a causal reading of a correlation:

- Selection bias: people choose whether to get the treatment, and those who choose it differ from those who do not in ways that also affect the outcome
- Reverse causality: the outcome $y$ also affects the explanatory variable $x$
- Simultaneity: $x$ and $y$ are determined together, for example in a market equilibrium

With selection bias, the difference that we observe between the two groups is a mix of two parts:

$$\text{observed difference} = \text{causal effect} + \text{selection bias}$$

From a simple comparison of the two groups, we cannot tell how much of the observed difference comes from each part.
To separate them, economists use randomized experiments or natural experiments (also called quasi-experiments), in which chance, or something else unrelated to the other factors that affect the outcome, determines who gets the treatment.
Today we'll discuss the first two threats.

When a factor in $u_i$ affects the outcome and is correlated with the treatment, the estimate is biased.
We can find the sign of the bias by answering two questions:

1. Does the omitted factor raise or lower the outcome?
2. Do treated people have more or less of it?

Multiply the two signs.
If both are positive or both are negative, the bias is positive: the estimate is larger than the causal effect.
If one is positive and the other negative, the bias is negative: the estimate is smaller than the causal effect.

🤔 Is 0.92 marks the effect of watching one more hour of recordings?
Use the two questions on two factors in $u_i$ in the regression of `quiz` on `hours`:

- Coming to the live lectures in person
- General study effort

For each factor, predict the sign of the bias on the coefficient on `hours`.

## Threats to causality

### Question 2: what's the threat?

For each scenario, name the main threat to a causal reading and the story behind it.

🤔 Scenario (a), gym membership and BMI (body mass index).
A researcher compares adults with and without gym memberships and finds that gym members have a much lower BMI, on average.
They estimate

$$\text{BMI}_i = \beta_0 + \beta_1 \text{gym}_i + u_i$$

where $\text{gym}_i$ equals 1 for gym members, and they conclude that joining a gym lowers BMI.
What's the threat, and what's the story?

🤔 Scenario (b), social media followers and sales.
A company analyst estimates

$$\text{sales}_i = \beta_0 + \beta_1 \text{followers}_i + u_i$$

across the company's products and finds $\hat{\beta}_1 > 0$: products with more social media followers have higher monthly sales.
The analyst tells management that more followers will raise sales, and proposes that the company buy followers.
What's the problem with this proposal?

## Which way does the bias go?

### Question 3: the sign of the bias

We'll use the two questions from question 1 on the gym scenario.

We'll skip the followers scenario here.
The two questions are about a third factor that affects both the treatment and the outcome, and reverse causality has none: the outcome itself, sales, affects the treatment, followers.

🤔 Gym membership and BMI: name at least two factors in $u_i$ that could explain why gym members have a lower BMI, even if the gym itself did nothing.
For each factor, answer the two questions and predict the sign of the bias on $\hat{\beta}_1$.